In [1]:
!pip install -q -U huggingface_hub transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 22.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.


In [2]:
%pip install -q antropy

Note: you may need to restart the kernel to use updated packages.


In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
import zipfile
import os
import shutil
import gc
import re

import antropy as ant
import numpy as np
import pandas as pd
from mne.filter import filter_data
from numba import njit, prange
from scipy.signal import hilbert, welch
from scipy.stats import entropy, kurtosis, skew

In [4]:
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("both_datasets")
login(token=HF_TOKEN)

In [5]:
# ==========================================================================
# 0. CONSTANTS AND PATHS
# ==========================================================================
FS = 256

BANDS = {
    'delta': [0.5, 4.0],
    'theta': [4.0, 8.0],
    'alpha': [8.0, 13.0],
    'beta':  [13.0, 30.0],
    'gamma': [30.0, 100],
}
MIN_EPOCHS = 1000

# Traditional features: <FEATURE_OUT>/<subject_id>/...csv
FEATURE_OUT = '/kaggle/working/Handcrafted'
# Connectivity: <CONN_OUT_*>/<PLI|wPLI>_matrix/<subject>/<group>/<file>/<band>.npy
CONN_OUT_TESTING = '/kaggle/working/CONNECTIVITY/Testing'    # used for FAR/h calculation
CONN_OUT_TRAINING = '/kaggle/working/CONNECTIVITY/Training'  # used for training / epoch-level metrics
PREICTAL_LABEL = 'preictal'

RAW_PARENTS = {'/kaggle/working/Training', '/kaggle/working/Testing'}

In [6]:
# ==========================================================================
# 1. CONNECTIVITY ESTIMATION FUNCTIONS (vectorised over epochs)
# ==========================================================================
# Input : analytic_signal (n_epochs, n_ch, n_times), complex
# Output: (n_epochs, n_ch, n_ch)
# Only the loop over channel pairs remains; each iteration handles all epochs at once.

def get_wpli_matrix(analytic_signal):
    n_ep, n_ch, _ = analytic_signal.shape
    phase = np.angle(analytic_signal)
    wpli_matrix = np.zeros((n_ep, n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            sin_dphi = np.sin(phase[:, i] - phase[:, j])          # (n_ep, n_times)
            wpli = np.abs(np.mean(sin_dphi, axis=-1)) / (np.mean(np.abs(sin_dphi), axis=-1) + 1e-12)
            wpli_matrix[:, i, j] = wpli_matrix[:, j, i] = wpli
    return wpli_matrix


def get_pli_matrix(analytic_signal):
    n_ep, n_ch, _ = analytic_signal.shape
    phase = np.angle(analytic_signal)
    pli_matrix = np.zeros((n_ep, n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            pli = np.abs(np.mean(np.sign(phase[:, i] - phase[:, j]), axis=-1))
            pli_matrix[:, i, j] = pli_matrix[:, j, i] = pli
    return pli_matrix


# Every entry here is computed and saved for each band
CONNECTIVITY_FUNCS = {
    'PLI': get_pli_matrix,
    'wPLI': get_wpli_matrix,
}

In [7]:
# ==========================================================================
# 2. TRADITIONAL FEATURE FUNCTIONS (vectorised over epochs AND channels)
# ==========================================================================
# Every function takes x of shape (n_epochs, n_channels, n_samples)
# and returns an array of shape (n_epochs, n_channels).

FEATURE_NAMES = [
    'mean', 'std', 'skewness', 'kurtosis',
    'mean_spectral_power', 'max_spectral_power', 'freq_of_max_power',
    'zero_crossing_rate', 'shannon_entropy', 'sample_entropy',
]

def shannon_entropy_batch(x, bins=10):
    """Same result as entropy(np.histogram(sig, bins=10, density=True)[0] + 1e-12) for every signal.

    The histogram is rebuilt with the same binning rules as np.histogram
    (uniform bins over [min, max], right-most bin closed), but for all signals at once.
    """
    n_ep, n_ch, n_t = x.shape
    lo, hi = x.min(axis=-1), x.max(axis=-1)
    flat = lo == hi                                  # np.histogram widens a degenerate range by +-0.5
    lo, hi = np.where(flat, lo - 0.5, lo), np.where(flat, hi + 0.5, hi)

    edges = lo[..., None] + np.arange(bins + 1) * ((hi - lo) / bins)[..., None]
    edges[..., -1] = hi

    idx = ((x - lo[..., None]) * (bins / (hi - lo))[..., None]).astype(np.intp)
    idx[idx == bins] -= 1
    idx[x < np.take_along_axis(edges, idx, axis=-1)] -= 1                                  # float-error fix
    idx[(x >= np.take_along_axis(edges, idx + 1, axis=-1)) & (idx != bins - 1)] += 1       # float-error fix

    offset = (np.arange(n_ep * n_ch) * bins).reshape(n_ep, n_ch, 1)
    counts = np.bincount((idx + offset).ravel(), minlength=n_ep * n_ch * bins).reshape(n_ep, n_ch, bins)

    density = counts / np.diff(edges, axis=-1) / counts.sum(axis=-1, keepdims=True)
    return entropy(density + 1e-12, axis=-1)

def sample_entropy_batch(x, order=2):
    n_ep, n_ch, _ = x.shape
    x64 = x.astype(np.float64)
    r = 0.2 * np.std(x64, axis=-1).reshape(-1)
    rows = np.ascontiguousarray(x64.reshape(n_ep * n_ch, -1))
    return _sample_entropy_rows(rows, order, r).reshape(n_ep, n_ch)

@njit
def _sample_entropy_1d(seq, order, r):
    """Identical to antropy._numba_sampen (Chebyshev metric), made callable from prange."""
    size = seq.size
    numerator = 0
    denominator = 0
    for offset in range(1, size - order):
        n_numerator = int(abs(seq[order] - seq[order + offset]) >= r)
        n_denominator = 0
        for idx in range(order):
            d = int(abs(seq[idx] - seq[idx + offset]) >= r)
            n_numerator += d
            n_denominator += d
        if n_numerator == 0:
            numerator += 1
        if n_denominator == 0:
            denominator += 1
        prev_in_diff = int(abs(seq[order] - seq[offset + order]) >= r)
        for idx in range(1, size - offset - order):
            out_diff = int(abs(seq[idx - 1] - seq[idx + offset - 1]) >= r)
            in_diff = int(abs(seq[idx + order] - seq[idx + offset + order]) >= r)
            n_numerator += in_diff - out_diff
            n_denominator += prev_in_diff - out_diff
            prev_in_diff = in_diff
            if n_numerator == 0:
                numerator += 1
            if n_denominator == 0:
                denominator += 1
    if denominator == 0:
        return np.nan
    elif numerator == 0:
        return np.inf
    else:
        return -np.log(numerator / denominator)


@njit(parallel=True)
def _sample_entropy_rows(X, order, r):
    out = np.empty(X.shape[0])
    for k in prange(X.shape[0]):          # every (epoch, channel) signal is independent -> multi-threaded
        out[k] = _sample_entropy_1d(X[k], order, r[k])
    return out

def extract_features_batch(x, fs=FS):
    """Traditional features for ALL epochs and channels at once.

    x: (n_epochs, n_channels, n_samples)
    Returns (n_epochs, n_channels * n_features), columns ordered channel-major:
    [mean_0, std_0, ..., sample_entropy_0, mean_1, ...] (see feature_columns).
    """
    n_t = x.shape[-1]
    freqs, psd = welch(x, fs=fs, nperseg=min(n_t, 256), axis=-1)       # psd: (n_ep, n_ch, n_freqs)

    feats = {
        'mean': x.mean(axis=-1),
        'std': x.std(axis=-1),
        'skewness': skew(x, axis=-1),
        'kurtosis': kurtosis(x, axis=-1),
        'mean_spectral_power': psd.mean(axis=-1),
        'max_spectral_power': psd.max(axis=-1),
        'freq_of_max_power': freqs[psd.argmax(axis=-1)],
        'zero_crossing_rate': np.sum(np.diff(np.sign(x), axis=-1) != 0, axis=-1) / n_t,
        'shannon_entropy': shannon_entropy_batch(x),
        'sample_entropy': sample_entropy_batch(x),
    }
    stacked = np.stack([feats[name] for name in FEATURE_NAMES], axis=-1)   # (n_ep, n_ch, n_feat)
    return stacked.reshape(x.shape[0], -1)


def feature_columns(n_channels):
    return [f'{name}_{ch}' for ch in range(n_channels) for name in FEATURE_NAMES]

In [8]:
# ==========================================================================
# 3. PIPELINE HELPERS
# ==========================================================================
def natural_key(name):
    return [(0, int(t)) if t.isdigit() else (1, t) for t in re.findall(r'\d+|\D+', name)]

def list_npy(path):
    return sorted((f for f in os.listdir(path) if f.endswith('.npy')), key=natural_key)

def filter_band(epochs, low, high):
    """Band-pass FIR filter along the last axis. epochs: (n_epochs, n_channels, n_samples)."""
    return filter_data(epochs, sfreq=FS, l_freq=low, h_freq=high,
                       method='fir', fir_design='firwin', verbose=False)


def save_connectivity(analytic_epochs, conn_root, rel_dir, band):
    """Compute (all epochs at once) and save every connectivity measure of CONNECTIVITY_FUNCS for one band."""
    for name, get_matrix in CONNECTIVITY_FUNCS.items():
        matrices = get_matrix(analytic_epochs)          # (n_epochs, n_channels, n_channels)
        out_dir = os.path.join(conn_root, f'{name}_matrix', rel_dir)
        os.makedirs(out_dir, exist_ok=True)
        np.save(os.path.join(out_dir, f'{band}.npy'), matrices)
        del matrices


def extract_band_features(filtered_epochs, band, seizure_id, epoch_offset=0, label=None):
    """Feature table (one row per epoch) for one band, computed on all epochs at once.

    Column order: features..., Band, SeizureID, Epoch, [Class]
    """
    n_epochs, n_ch, _ = filtered_epochs.shape
    df = pd.DataFrame(extract_features_batch(filtered_epochs), columns=feature_columns(n_ch))
    df['Band'] = band
    df['SeizureID'] = seizure_id
    df['Epoch'] = epoch_offset + np.arange(n_epochs)
    if label is not None:
        df['Class'] = label
    return df


def process_npy_file(npy_path, conn_root, rel_dir, seizure_id, epoch_offset=0, label=None):
    """Filter each band once, then compute connectivity and traditional features from it.

    Returns (feature_df, n_epochs).
    """
    epochs = np.load(npy_path).astype(np.float64)  # (n_epochs, n_channels, n_samples)
    dfs = []
    for band, (low, high) in BANDS.items():
        filtered = filter_band(epochs, low, high)
        analytic = hilbert(filtered, axis=-1)
        save_connectivity(analytic, conn_root, rel_dir, band)
        dfs.append(extract_band_features(filtered, band, seizure_id, epoch_offset, label))
        del filtered, analytic
    n_epochs = epochs.shape[0]
    del epochs
    gc.collect()
    return pd.concat(dfs, ignore_index=True), n_epochs


def save_features(dfs, subject_id, file_name):
    if not dfs:
        print(f'[WARN] no data -> {subject_id}/{file_name} NOT saved')
        return
    out_dir = os.path.join(FEATURE_OUT, subject_id)
    os.makedirs(out_dir, exist_ok=True)
    pd.concat(dfs, ignore_index=True).to_csv(os.path.join(out_dir, file_name), index=False)

In [9]:
def load_training_dataset_chb(chb):
    zip_path = hf_hub_download(
        repo_id="Lann000/CHB-MIT_Segmentation_Training",
        filename=f"{chb}.zip",
        repo_type="dataset",
        token=HF_TOKEN
    )
    extract_dir = f"/kaggle/working/Training/{chb}"
    
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    return extract_dir

def load_testing_dataset_chb(chb):
    zip_path = hf_hub_download(
        repo_id="Lann000/CHB-MIT_Segmentation_Testing",
        filename=f"{chb}.zip",
        repo_type="dataset",
        token=HF_TOKEN
    )
    extract_dir = f"/kaggle/working/Testing/{chb}"
    
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    return extract_dir

def remove_chb_dir(folder):
    """Chỉ xóa thư mục dạng /kaggle/working/{Training|Testing}/<chb>."""
    folder = os.path.abspath(folder)
    if os.path.dirname(folder) not in RAW_PARENTS:
        raise ValueError(f'Refuse to delete: {folder}')
    shutil.rmtree(folder)
    print(f'Removed {folder}')

In [10]:
# ==========================================================================
# 4. SUBJECT-LEVEL PROCESSING
# ==========================================================================
def select_last_files(block_path, min_epochs=MIN_EPOCHS):
    selected, total = [], 0
    for fname in reversed(list_npy(block_path)):
        n = np.load(os.path.join(block_path, fname), mmap_mode='r').shape[0]
        selected.append(fname)
        total += n
        if total >= min_epochs:
            break
    return selected[::-1], total

def process_testing_subject(folder):
    subject_id = folder[-5:]
    print(f'Processing {subject_id} (testing)')

    for sz in sorted(d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))):
        sz_path = os.path.join(folder, sz)
        blocks = sorted((d for d in os.listdir(sz_path) if os.path.isdir(os.path.join(sz_path, d))),
                key=natural_key)
        if not blocks:
            print(f'[WARN] {subject_id}/{sz}: no block folders, skipped')
            continue

        block = blocks[-1]
        block_path = os.path.join(sz_path, block)
        print(f'  {sz}: using {block}')

        files, total = select_last_files(block_path)
        if total < MIN_EPOCHS:
            print(f'[WARN] {subject_id}/{sz}/{block}: chỉ có {total} epoch (< {MIN_EPOCHS})')

        epoch_offset = 0
        dfs = []
        for fname in files:
            stem = os.path.splitext(fname)[0]
            df, n_epochs = process_npy_file(
                os.path.join(block_path, fname), CONN_OUT_TESTING,
                os.path.join(subject_id, sz, block, stem),
                seizure_id=sz, epoch_offset=epoch_offset)
            df['Block'] = block
            dfs.append(df)
            epoch_offset += n_epochs

        save_features(dfs, subject_id, f'{sz}_{block}_{subject_id}.csv')
        del dfs
        gc.collect()

def process_training_subject(folder):
    subject_id = folder[-5:]
    print(f'Processing {subject_id} (training)')

    for label in sorted(d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))):
        label_path = os.path.join(folder, label)
        
        if label == PREICTAL_LABEL:
            dfs = []
            for fname in list_npy(label_path):
                stem = os.path.splitext(fname)[0]
                df, _ = process_npy_file(
                    os.path.join(label_path, fname), CONN_OUT_TRAINING,
                    os.path.join(subject_id, label, stem), seizure_id=stem, label=label)
                dfs.append(df)

            save_features(dfs, subject_id, f'{label}.csv')
            del dfs
        else:
            for sz in sorted(d for d in os.listdir(label_path) if os.path.isdir(os.path.join(label_path, d))):
                sz_path = os.path.join(label_path, sz)
                epoch_offset = 0
                dfs = []
                for fname in list_npy(sz_path):
                    stem = os.path.splitext(fname)[0]
                    df, n_epochs = process_npy_file(
                        os.path.join(sz_path, fname), CONN_OUT_TRAINING,
                        os.path.join(subject_id, label, sz, stem),
                        seizure_id=sz, epoch_offset=epoch_offset, label=label)
                    dfs.append(df)
                    epoch_offset += n_epochs
                save_features(dfs, subject_id, f'{sz}_{label}.csv')
                del dfs
        gc.collect()

In [11]:
# ==========================================================================
# 5. RUN
# ==========================================================================
RUN_TESTING = True
RUN_TRAINING = True
chbs = ['chb04']
#for chb in chbs[0:1]: #chb01     #Run each chb in one version
#for chb in chbs[1:2]: #chb02
#for chb in chbs[2:3]: #chb03
#for chb in chbs[3:4]: #chb05
for chb in chbs[4:5]: #chb06
#for chb in chbs[5:6]: #chb07
#for chb in chbs[6:7]: #chb09
#for chb in chbs[7:8]: #chb10
#for chb in chbs[8:9]: #chb13
#for chb in chbs[9:10]: #chb14
#for chb in chbs[10:11]: #chb18
#for chb in chbs[11:12]: #chb20
#for chb in chbs[12:13]: #chb21
#for chb in chbs[13:14]: #chb22
#for chb in chbs[14:15]: #chb23
    if RUN_TESTING:
        folder = load_testing_dataset_chb(chb)
        try:
            process_testing_subject(folder)
        finally:
            remove_chb_dir(folder)
    if RUN_TRAINING:
        folder = load_training_dataset_chb(chb)
        try:
            process_training_subject(folder)
        finally:
            remove_chb_dir(folder)